## Lab 04 – argumenty funkcji i dekoratory

1. Argumenty tylko po nazwie – `flow_rate`
2. Pomiar czasu – `@timeit`
3. Pochodna numeryczna – `@derivate`
4. Sprawdzanie typów argumentów – `@accepts`
5. Ponawianie po błędzie – `@retry`
6. Zapamiętywanie wyników – `@cached`
7. Złożenie funkcji – `@zlozenie`
8. Obsługa błędów: `match` i rejestr `@error_handler`

Zadanie dodatkowe: Decode Ways (LeetCode 91)

### Przypomnienie: dekorator

Dekorator to funkcja, która przyjmuje funkcję i zwraca nową funkcję – zwykle „opakowanie”, które robi coś przed lub po wywołaniu oryginału. Zapis `@dekorator` nad `def f` to skrót od `f = dekorator(f)`.

Dekorator **z parametrami** (`@dekorator(3)`) ma jeden poziom więcej: `dekorator(3)` zwraca dopiero właściwy dekorator.

In [ ]:
import functools


def shout(func):
    @functools.wraps(func)  # kopiuje nazwę i docstring z func na wrapper
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper


def repeat(n):  # dekorator z parametrem
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(n)]
        return wrapper
    return decorator


@shout
def greet(name):
    """Wita się"""
    return f"cześć, {name}"


@repeat(3)
def ping():
    return "ping"


print(greet("ala"))
print(ping())
print(greet.__name__, "-", greet.__doc__)

## Zadanie 1

Funkcja `flow_rate` liczy, ile produktu przybywa w jednostce czasu – na razie zawsze w kilogramach na sekundę. Chcemy móc dostać wynik np. w gramach na minutę, ale bez psucia istniejących wywołań.

Zmodyfikuj `flow_rate` tak, aby:
* przyjmowała dwa opcjonalne parametry: `period` (długość okresu w sekundach, domyślnie `1`) i `units_per_kg` (ile jednostek przypada na kilogram, domyślnie `1`),
* stare wywołanie `flow_rate(weight, time)` działało bez zmian,
* nowe parametry dało się podać **tylko po nazwie** (zobacz [argumenty keyword-only](https://docs.python.org/3/tutorial/controlflow.html#special-parameters)).

In [ ]:
def flow_rate(weight, time):
    """Zwraca, ile produktu (w kg) przybyło w jednostce czasu (w sekundach)"""
    return weight / time

In [ ]:
import math


print(math.isclose(flow_rate(0.5, 3), 0.5 / 3))                                       # kg/s
print(math.isclose(flow_rate(0.5, 3, period=60, units_per_kg=1000), 10_000))          # g/min
print(math.isclose(flow_rate(0.5, 3, period=3600), 600))                              # kg/h

try:
    flow_rate(0.5, 3, 3600, 2.2)
    print(False)
except TypeError:
    print(True)

> **Do przemyślenia:** dlaczego to dobrze, że ostatnie wywołanie rzuca `TypeError`? Co oznaczają `3600` i `2.2` dla kogoś, kto czyta ten kod?

<details>
<summary>Odpowiedź</summary>

Z samego wywołania `flow_rate(0.5, 3, 3600, 2.2)` nie da się odgadnąć, co znaczą liczby `3600` i `2.2` – trzeba zajrzeć do definicji funkcji. Łatwo też pomylić kolejność i po cichu dostać zły wynik. Wymuszenie nazw (`period=3600, units_per_kg=2.2`) sprawia, że wywołanie jest czytelne samo w sobie, a pomyłka w kolejności jest niemożliwa. Dodatkowo autor funkcji może później zmienić kolejność tych parametrów albo dodać nowe, nie psując niczyjego kodu.

</details>

## Zadanie 2

Napisz dekorator `timeit`, który:
* wywołuje udekorowaną funkcję i zwraca jej wynik,
* wypisuje czas wykonania w postaci `Function NAZWA took: 0.00001 seconds`,
* zachowuje nazwę i docstring funkcji (`functools.wraps`, jak w przypomnieniu na górze).

Do mierzenia czasu użyj [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter) – jest dokładniejszy niż `time.time`.

In [ ]:
import functools
from time import perf_counter


def timeit(func):
    """Wypisuje czas wykonania udekorowanej funkcji"""
    pass

In [ ]:
@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares


@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]


@timeit
def squares_map(n):
    return map(lambda x: x ** 2, range(n))


print(squares_list(5) == [0, 1, 4, 9, 16])
print(list(squares_map(5)) == [0, 1, 4, 9, 16])
print(squares_comprehension.__name__ == "squares_comprehension")

In [ ]:
n = 1_000_000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

> **Do przemyślenia:** dlaczego `squares_map` jest „najszybsze”? Czy na pewno policzyło kwadraty?

<details>
<summary>Odpowiedź</summary>

Nie policzyło. W Pythonie 3 `map` jest **leniwe** – zwraca iterator, który liczy kolejne elementy dopiero wtedy, gdy ktoś po nie sięgnie (np. `list(m)` albo pętla `for`). Dekorator zmierzył więc tylko utworzenie obiektu `map`. Po wymuszeniu obliczeń (`list(map(...))`) wersja z `lambda` jest zwykle nawet trochę wolniejsza od list comprehension, bo dla każdego elementu wywołuje funkcję.

</details>

## Zadanie 3

Pochodną funkcji można przybliżyć ilorazem różnicowym: $f'(x) \approx \frac{f(x+h) - f(x)}{h}$ dla małego $h$.

Napisz dekorator z parametrem `derivate(h=None)`, który zamienia funkcję $f$ w (przybliżoną) funkcję $f'$:
* `h` to krok ilorazu różnicowego,
* jeśli `h` nie zostanie podane, przyjmij `1000 * sys.float_info.epsilon` (epsilon maszynowy).

Uwaga: dekorator z parametrem domyślnym i tak wywołujemy z nawiasami – `@derivate()`.

In [ ]:
import sys


def derivate(h=None):
    """Zamienia funkcję f w przybliżenie jej pochodnej f'"""
    pass

In [ ]:
import math


@derivate(0.0001)
def f(x):
    return x * x            # f'(x) = 2x


@derivate(0.00001)
def g(x):
    return x ** 3 + 3       # g'(x) = 3x^2


@derivate()
def s(x):
    return math.sin(x)      # s'(x) = cos(x)


print(math.isclose(f(100), 200, rel_tol=1e-3))
print(math.isclose(f(-3), -6, rel_tol=1e-3))
print(math.isclose(g(2), 12, rel_tol=1e-3))
print(math.isclose(s(0), 1, rel_tol=1e-3))

## Zadanie 4

Python nie sprawdza typów argumentów w trakcie działania programu. Napisz dekorator `accepts(*types)`, który to robi:
* `types` to oczekiwane typy kolejnych parametrów funkcji,
* jeśli któryś argument ma zły typ, dekorator rzuca `TypeError`,
* sprawdzanie ma działać także dla argumentów podanych po nazwie.

Wskazówka: `inspect.signature(func).bind(*args, **kwargs).arguments` zwraca słownik `{nazwa_parametru: wartość}` w kolejności parametrów.

In [ ]:
import inspect


def accepts(*types):
    """Sprawdza, czy argumenty udekorowanej funkcji mają typy podane w `types`"""
    pass

In [ ]:
@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]


@accepts(float, int)
def static_pow(base, exp):
    return base ** exp


print(capitalize("ola") == "Ola")
print(static_pow(2.0, 2) == 4.0)
print(static_pow(2.0, exp=2) == 4.0)
print(static_pow(base=2.0, exp=2) == 4.0)

for call in [lambda: capitalize(2), lambda: static_pow("x", 10), lambda: static_pow(2.0, 2.2)]:
    try:
        call()
        print(False)
    except TypeError:
        print(True)

## Zadanie 5

Operacje sieciowe (pobieranie stron, zapytania do API, połączenia z bazą danych) potrafią losowo się nie udać, a ponowienie próby często wystarcza. Zamiast otaczać każde wywołanie pętlą z `try`/`except`, można napisać do tego dekorator.

Napisz dekorator `retry(times=3, exceptions=(Exception,))`, który:
* wywołuje udekorowaną funkcję i zwraca jej wynik,
* jeśli funkcja rzuci wyjątek z krotki `exceptions`, ponawia wywołanie – łącznie co najwyżej `times` prób,
* jeśli wszystkie próby się nie udadzą, rzuca wyjątek z ostatniej próby,
* wyjątków spoza `exceptions` nie łapie – lecą od razu, bez ponawiania,
* zachowuje nazwę i docstring funkcji.

*(dla chętnych)* Dodaj parametr `delay` – czas oczekiwania między próbami (`time.sleep`), podwajany po każdej nieudanej próbie (tzw. *exponential backoff*). W praktyce używa się do tego gotowych bibliotek, np. [`tenacity`](https://tenacity.readthedocs.io/).

In [ ]:
import functools


def retry(times=3, exceptions=(Exception,)):
    """Ponawia wywołanie udekorowanej funkcji, gdy rzuci ona jeden z wyjątków `exceptions`"""
    pass

In [ ]:
def make_flaky(failures, exception=ConnectionError):
    """Zwraca funkcję, która rzuca `exception` przy pierwszych `failures` wywołaniach, a potem zwraca "OK",
    oraz słownik z liczbą wywołań"""
    calls = {"n": 0}

    def fetch_data():
        calls["n"] += 1
        if calls["n"] <= failures:
            raise exception(f"próba {calls['n']} nieudana")
        return "OK"

    return fetch_data, calls


# 2 porażki, 3 próby -> sukces za trzecim razem
fetch, calls = make_flaky(failures=2)
print(retry(times=3)(fetch)() == "OK" and calls["n"] == 3)

# 5 porażek, 3 próby -> wyjątek po trzeciej próbie
fetch, calls = make_flaky(failures=5)
try:
    retry(times=3)(fetch)()
    print(False)
except ConnectionError:
    print(calls["n"] == 3)

# wyjątek spoza `exceptions` -> bez ponawiania
fetch, calls = make_flaky(failures=1, exception=ValueError)
try:
    retry(times=3, exceptions=(ConnectionError, TimeoutError))(fetch)()
    print(False)
except ValueError:
    print(calls["n"] == 1)


@retry(times=2)
def download(url):
    """Pobiera stronę"""
    return f"<html>{url}</html>"

print(download("uj.edu.pl") == "<html>uj.edu.pl</html>")
print(download.__name__ == "download" and download.__doc__ == "Pobiera stronę")

## Zadanie 6

*Cache* (pamięć podręczna) zapamiętuje wyniki wywołań, żeby nie liczyć ich drugi raz. Napisz dekorator `cached`, który:
* dla danych argumentów wywołuje funkcję tylko raz, a przy kolejnych wywołaniach zwraca zapamiętany wynik,
* działa dla dowolnej liczby argumentów pozycyjnych i nazwanych; kolejność argumentów nazwanych nie ma znaczenia,
* nie musi utożsamiać `foo(3)` z `foo(x=3)` ani z wywołaniem z jawnie podanymi wartościami domyślnymi,
* dodaje do opakowanej funkcji metodę `.cache_reset()`, która czyści cache,
* dodaje metodę `.cache_status()`, która zwraca napis `Function NAZWA called X times, evaluated Y times` (liczniki nie są zerowane przez `cache_reset`).

Wskazówka: funkcja to obiekt – można przypisać jej atrybut, np. `wrapper.cache_reset = ...`.

In [ ]:
def cached(func):
    """Zapamiętuje wyniki wywołań udekorowanej funkcji"""
    pass

In [ ]:
from random import random


@cached
def foo(x, y=1, z=4):
    return random()


print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == "Function foo called 10 times, evaluated 5 times")

<details>
<summary><b>Ciekawostka:</b> pułapka z zasięgiem zmiennych w domknięciach</summary>

Gdy w zadaniu 6 liczniki wywołań trzymasz w zwykłej zmiennej, łatwo trafić na taki błąd:

```python
def make_counter():
    count = 0
    def counter():
        count += 1      # UnboundLocalError!
        return count
    return counter

make_counter()()
```
```
UnboundLocalError: local variable 'count' referenced before assignment
```

Python już podczas kompilowania funkcji `counter` widzi przypisanie do `count`, więc uznaje `count` za zmienną **lokalną** tej funkcji – a `count += 1` próbuje ją odczytać, zanim dostała wartość. Rozwiązaniem jest `nonlocal count` na początku `counter`. Słownik (`calls["n"] += 1`) nie ma tego problemu, bo nie przypisujemy nazwy na nowo, tylko zmieniamy zawartość obiektu – dlatego `make_flaky` w zadaniu 5 trzyma licznik w słowniku.

*Więcej: Fluent Python (2. wyd.), rozdz. 9.*

</details>

<details>
<summary><b>Ciekawostka:</b> gotowy dekorator `functools.cache`</summary>

Biblioteka standardowa ma gotowy dekorator [`functools.cache`](https://docs.python.org/3/library/functools.html#functools.cache). Rekurencyjny Fibonacci bez cache dla `n = 100` nigdy by się nie skończył, a z cache liczy każdą wartość tylko raz:

```python
from functools import cache


@cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(100))
print(fib.cache_info())
```
```
354224848179261915075
CacheInfo(hits=98, misses=101, maxsize=None, currsize=101)
```

`misses=101` to liczba faktycznie policzonych wartości (od `fib(0)` do `fib(100)`), a `hits=98` – ile razy wynik wzięto z pamięci.

</details>

## Zadanie 7

Napisz dekorator z parametrem `zlozenie(n)`, który zamienia funkcję $f$ w jej $n$-krotne złożenie, czyli $f(f(\dots f(x)))$.

In [ ]:
def zlozenie(n):
    """Zamienia funkcję f w jej n-krotne złożenie"""
    pass

In [ ]:
@zlozenie(3)
def f1(x):
    return x + 1


@zlozenie(2)
def f2(x):
    return x * x


@zlozenie(5)
def f3(word):
    return "".join(chr(ord(letter) + 1) for letter in word)


print(f1(2) == 5)
print(f2(3) == 81)
print(f3("alamakota") == "fqfrfptyf")

## Zadanie 8

Programy często muszą zamienić wyjątek na zrozumiały komunikat dla użytkownika. Poniższa komórka przygotowuje kilka prawdziwych wyjątków, na których będziemy pracować – uruchom ją.

In [ ]:
def capture(func):
    """Wywołuje func i zwraca rzucony przez nią wyjątek"""
    try:
        func()
    except Exception as error:
        return error


errors = {
    "zero": capture(lambda: 1 / 0),
    "file": capture(lambda: open("nie_ma_takiego_pliku.txt")),
    "value": capture(lambda: int("abc")),
    "type": capture(lambda: "a" + 1),
    "key": capture(lambda: {}["x"]),
}

for name, error in errors.items():
    print(f"{name:6} {type(error).__name__:18} {error}")

a) Od Pythona 3.10 jest instrukcja [`match`](https://docs.python.org/3/tutorial/controlflow.html#match-statements) – odpowiednik `switch` z innych języków, ale potrafi więcej. Między innymi dopasowuje obiekty po klasie:
* `case ZeroDivisionError():` pasuje do każdego obiektu tej klasy (i jej podklas),
* `case FileNotFoundError(filename=name):` dodatkowo przypisuje atrybut `filename` do zmiennej `name`,
* `case ValueError() | TypeError():` pasuje do jednej z kilku klas,
* `case _:` pasuje do wszystkiego (jak `default`).

Napisz funkcję `error_message(error)`, która przy pomocy `match` zwraca:
* dla `ZeroDivisionError` – `"Nie można dzielić przez zero"`,
* dla `FileNotFoundError` – `"Nie znaleziono pliku: NAZWA"`,
* dla `ValueError` lub `TypeError` – `"Niepoprawne dane"`,
* dla pozostałych – `"Nieznany błąd"`.

In [ ]:
def error_message(error):
    """Zamienia wyjątek na komunikat dla użytkownika"""
    pass

In [ ]:
print(error_message(errors["zero"]) == "Nie można dzielić przez zero")
print(error_message(errors["file"]) == "Nie znaleziono pliku: nie_ma_takiego_pliku.txt")
print(error_message(errors["value"]) == "Niepoprawne dane")
print(error_message(errors["type"]) == "Niepoprawne dane")
print(error_message(errors["key"]) == "Nieznany błąd")

b) Gdy rodzajów błędów przybywa, a obsługę dopisują różne części programu, jeden duży `match` staje się trudny w utrzymaniu. Zamiast tego można trzymać **rejestr** funkcji obsługujących poszczególne typy błędów – tak działa np. [`@app.errorhandler`](https://flask.palletsprojects.com/en/stable/errorhandling/) we Flasku.

Napisz:
* dekorator `error_handler(exc_type)`, który zapisuje udekorowaną funkcję w słowniku `HANDLERS` pod kluczem `exc_type` i zwraca ją **bez zmian** – dekorator nie musi niczego opakowywać,
* funkcję `handle(error)`, która wywołuje funkcję obsługującą typ błędu; jeśli dla danego typu jej nie ma, szuka jej dla klas bazowych (np. `FileNotFoundError` dziedziczy po `OSError`), a gdy nic nie znajdzie, zwraca `"Nieznany błąd"`.

Wskazówka: `type(error).__mro__` to krotka z klasą błędu i wszystkimi jej klasami bazowymi, od najbardziej szczegółowej.

In [ ]:
HANDLERS = {}


def error_handler(exc_type):
    """Rejestruje udekorowaną funkcję jako obsługę błędów typu `exc_type`"""
    pass


def handle(error):
    """Wywołuje funkcję obsługującą typ błędu (lub jego klasę bazową)"""
    pass

In [ ]:
@error_handler(ZeroDivisionError)
def division_by_zero(error):
    return "Nie można dzielić przez zero"


@error_handler(OSError)
def os_error(error):
    return f"Błąd systemu plików: {error.filename}"


@error_handler(ValueError)
def bad_value(error):
    return "Niepoprawne dane"


print(set(HANDLERS) == {ZeroDivisionError, OSError, ValueError})
print(division_by_zero(None) == "Nie można dzielić przez zero")  # funkcja nie została opakowana
print(handle(errors["zero"]) == "Nie można dzielić przez zero")
print(handle(errors["file"]) == "Błąd systemu plików: nie_ma_takiego_pliku.txt")  # przez klasę bazową OSError
print(handle(errors["value"]) == "Niepoprawne dane")
print(handle(errors["key"]) == "Nieznany błąd")

> **Do przemyślenia:** kiedy lepszy jest `match` z punktu a), a kiedy rejestr z punktu b)?

<details>
<summary>Odpowiedź</summary>

`match` jest czytelniejszy, gdy wszystkie przypadki są znane z góry i mieszczą się w jednym miejscu – całą logikę widać od razu. Rejestr sprawdza się, gdy przypadki dochodzą z różnych miejsc programu (np. każdy moduł albo plugin rejestruje obsługę swoich błędów), bo dodanie nowego nie wymaga zmiany funkcji `handle`.

</details>

<details>
<summary><b>Ciekawostka:</b> dekoratory działają już przy definicji funkcji</summary>

Dekorator jest wywoływany w momencie wykonania `def`, a nie przy wywołaniu udekorowanej funkcji – dla kodu w module oznacza to: **przy imporcie**.

```python
REGISTRY = []

def register(func):
    print(f"rejestruję {func.__name__}")
    REGISTRY.append(func)
    return func

@register
def f1():
    print("wywołanie f1")

print("REGISTRY:", [f.__name__ for f in REGISTRY])
f1()
```
```
rejestruję f1
REGISTRY: ['f1']
wywołanie f1
```

Dlatego w punkcie b) słownik `HANDLERS` jest wypełniony, zanim ktokolwiek wywoła `handle`, a frameworki takie jak Flask „znają” wszystkie trasy zaraz po zaimportowaniu modułu.

*Więcej: Fluent Python (2. wyd.), rozdz. 9.*

</details>

<details>
<summary><b>Ciekawostka:</b> gotowy rejestr według typu – `functools.singledispatch`</summary>

Biblioteka standardowa ma gotowy rejestr według typu – [`functools.singledispatch`](https://docs.python.org/3/library/functools.html#functools.singledispatch). Wybiera funkcję na podstawie typu pierwszego argumentu (także przez klasy bazowe), a typ można podać jako wskazówkę typu przy parametrze:

```python
from functools import singledispatch


@singledispatch
def describe(error):
    return "Nieznany błąd"


@describe.register
def _(error: ZeroDivisionError):
    return "Nie można dzielić przez zero"


@describe.register
def _(error: OSError):
    return f"Błąd systemu plików: {error.filename}"


for name in ["zero", "file", "key"]:
    print(f"{name:6} -> {describe(errors[name])}")
```
```
zero   -> Nie można dzielić przez zero
file   -> Błąd systemu plików: nie_ma_takiego_pliku.txt
key    -> Nieznany błąd
```

</details>

## Zadanie dodatkowe

To zadanie można zrobić **zamiast jednego z zadań 1–8**. Pochodzi z LeetCode: [Decode Ways (91)](https://leetcode.com/problems/decode-ways/).

Wiadomość złożoną z wielkich liter zakodowano, zamieniając każdą literę na jej numer w alfabecie: `A -> "1"`, `B -> "2"`, …, `Z -> "26"`. Ponieważ numery mają jedną albo dwie cyfry, ten sam ciąg cyfr można odkodować na kilka sposobów, np. `"226"` to `"BZ"` (2 26), `"VF"` (22 6) albo `"BBF"` (2 2 6).

Napisz funkcję `num_decodings(s)`, która zwraca, na ile sposobów można odkodować napis `s`. Uwaga na zera: `"06"` nie oznacza `F`, a samo `"0"` nie oznacza żadnej litery – takie fragmenty nie mają poprawnego odkodowania.

Ostatni test sprawdza napis ze 100 jedynek. Rozwiązanie, które przegląda wszystkie możliwości po kolei, nie skończy się w rozsądnym czasie – **trzeba je przyspieszyć tym, co było w tym labie.**

<details>
<summary>Podpowiedź 1</summary>

Spójrz na początek napisu. Można z niego wziąć jedną cyfrę (jeśli nie jest zerem) albo dwie cyfry (jeśli tworzą liczbę od 10 do 26). W obu przypadkach zostaje krótszy napis do odkodowania – ten sam problem, tylko mniejszy. Wynik to suma liczby sposobów dla obu przypadków.

</details>

<details>
<summary>Podpowiedź 2</summary>

Rekurencja z podpowiedzi 1 wielokrotnie liczy to samo dla tych samych końcówek napisu. Wystarczy zapamiętywać wyniki – dekoratorem `@cache` albo własnym `@cached` z zadania 6. Wygodnie jest, gdy funkcja rekurencyjna przyjmuje indeks `i` („odkoduj `s[i:]`”), a nie sam napis.

</details>

In [ ]:
def num_decodings(s):
    """Zwraca liczbę sposobów odkodowania napisu s (A=1, ..., Z=26)"""
    pass

In [ ]:
print(num_decodings("12") == 2)        # AB, L
print(num_decodings("226") == 3)       # BZ, VF, BBF
print(num_decodings("11106") == 2)     # AAJF, KJF
print(num_decodings("27") == 1)        # tylko BG – 27 to nie litera
print(num_decodings("10") == 1)        # tylko J
print(num_decodings("06") == 0)
print(num_decodings("100") == 0)
print(num_decodings("2101") == 1)      # BJA
print(num_decodings("1" * 100) == 573147844013817084101)